# Read from Bronze Table

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import StringType
from pyspark.sql.window import Window

In [0]:
raw_df = spark.table("workspace.bronze.crm_prd_info")
display(raw_df.limit(50))

# Data Transformations

In [0]:
rename_mapping = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}

df = raw_df

# Trimming
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(col(field.name)))

# Normalize production line
df = (
    df
    .withColumn("prd_line", F.trim("prd_line"))
    .withColumn(
        "prd_line", 
        F.when(col("prd_line") == "M", "Mountain")
        .when(col("prd_line") == "R", "Road")
        .when(col("prd_line") == "S", "Other Sales")
        .when(col("prd_line") == "T", "Touring")
        .otherwise("n/a"))
)

# Create cat_id and prd_key, fill missing prd_cost
df = df.withColumn("cat_id", F.replace(F.substring(col("prd_key"), 1, 5), F.lit("-"), F.lit("_")))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length("prd_key")))
df = df.fillna({"prd_cost": 0})
# df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

# Calculate prd_end_dt
window_spec = Window.partitionBy("prd_key").orderBy(col("prd_start_dt"))
df = df.withColumn("prd_end_dt", F.date_add(F.lead(col("prd_start_dt"), 1).over(window_spec), -1))

# Column names are not friendly
for old_name, new_name in rename_mapping.items():
    df = df.withColumnRenamed(old_name, new_name)

display(df.limit(50))

# Write into Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_products")

# Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_products LIMIT 20;